# **Tratamento de Dados e Pré-processamento**
Origem dos Dados:

O conjunto de dados utilizado nesta etapa foi extraído da plataforma Kaggle (Steam Games Dataset). Ele contém informações detalhadas sobre milhares de jogos publicados na loja virtual Steam.

Objetivo e Problema a ser Resolvido:

O objetivo deste projeto é prever o Preço de um jogo com base em suas características mercadológicas e de jogabilidade. Como a variável alvo (preço) é um valor numérico contínuo, este se caracteriza estritamente como um problema de Regressão.

Dados Rotulados (Variável Alvo) e Atributos: > * Variável Alvo (y):
A coluna referente ao preço do jogo (Price).

  Atributos Independentes (X): Serão utilizadas as características de cada título, que podem incluir o tempo médio de jogo, desenvolvedor, ano de lançamento, suporte a idiomas e categorias/gêneros (como Ação, RPG, Indie, Multiplayer).

Seleção e Tratamento Inicial dos Dados: > O dataset original possui um volume massivo de registros, incluindo milhares de jogos gratuitos (Free-to-Play). Como a precificação de jogos gratuitos obedece a modelos de negócios paralelos (como microtransações que não estão mapeadas nas variáveis independentes), eles introduziriam um forte viés matemático ao algoritmo.

Portanto, o tratamento de dados consistiu em:

Limpeza: Eliminação de linhas com o valor de preço nulo (dropna) e exclusão sumária de jogos gratuitos (preço igual a zero).

Amostragem (Sampling): Devido à alta complexidade computacional temporal de algoritmos baseados em distância (como o SVR - Support Vector Regressor), a base foi submetida a uma amostragem aleatória, limitando-se a 10.000 instâncias. Isso garante um volume robusto para o treinamento sem comprometer a viabilidade da execução da Validação Cruzada no hardware local.

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [ ]:
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.svm import SVR

In [ ]:
import kagglehub
import os

# Download latest version
path = kagglehub.dataset_download("fronkongames/steam-games-dataset")
print("Path to dataset files:", path)
caminho_csv = os.path.join(path, "games.csv")
dados_steam = pd.read_csv(caminho_csv)

In [ ]:
print(f"Tamanho original da base: {len(dados_steam)} jogos.")

In [ ]:
print("Valores nulos antes do tratamento:")
print(dados_steam.isnull().sum())

# **Tratamento de Dados e Valores Ausentes (Missing Values)**
Durante a análise exploratória, identificou-se uma alta incidência de valores ausentes em colunas como Reviews, Metacritic url e Score rank, além da presença de dezenas de colunas contendo apenas metadados e textos longos (como URLs, e-mails de suporte e descrições do jogo). Também foi notada uma grande quantidade de jogos com preço igual a zero (gratuitos).

A estratégia de tratamento adotada foi:



*   Exclusão de Atributos: As colunas com massiva quantidade de dados faltantes e os atributos puramente textuais ou identificadores (como AppID, Name, About the game, Website) foram removidas do modelo. A imputação de dados com alta vacância causaria forte viés, e textos brutos não podem ser processados diretamente por algoritmos baseados em distância e álgebra.
*   Filtro de Ruído: Jogos gratuitos  foram removidos da base, visto que a precificação deles obedece a modelos de negócios diferentes (microtransações) que não estão presentes nos atributos, o que distorceria severamente a previsão numérica do preço.
*   Eliminação de Linhas (Dropna): Para as variáveis essenciais (Price, Genres, Categories e Developers), aplicou-se a exclusão das linhas (dropna()) que continham valores nulos residuais, garantindo que os algoritmos regressivos (Regressão Linear / KNN Regressor / SVR / MLP) recebessem uma matriz densa e consistente para o treinamento.
*   Amostragem (Sampling): Para contornar a alta complexidade computacional quadrática de algoritmos como o SVR, a base tratada foi reduzida a uma amostra aleatória de 10.000 instâncias, mantendo a significância estatística e viabilizando a Validação Cruzada.

**Escalonamento de Características (Feature Scaling)**

Para preparar os dados para os algoritmos de regressão, aplicou-se a Padronização (Standardization) através da classe StandardScaler, rejeitando-se propositalmente o uso da Normalização.

A justificativa técnica para rejeitar a Normalização:

O mercado de jogos na Steam é altamente assimétrico e repleto de outliers extremos. Variáveis como Pico de Jogadores (Peak CCU) ou Horas Jogadas frequentemente atingem números na casa dos milhões para jogos virais. Se a Normalização tradicional fosse utilizada (limitando tudo estritamente entre 0 e 1), um único jogo viral forçaria todos os outros milhares de jogos comuns a assumirem valores infinitesimais (ex: 0.0001), destruindo a variância dos dados.

A Padronização ($z = \frac{x - \mu}{\sigma}$) resolve esse problema centralizando a média em zero e medindo o distanciamento em desvios padrão. Isso preserva a distância proporcional entre os jogos e garante o funcionamento correto de algoritmos baseados em cálculo de distância geométrica, como o SVR (Support Vector Regressor) e o KNN Regressor.

In [ ]:
colunas_para_remover = [
    'AppID', 'Name', 'About the game', 'Reviews', 'Header image',
    'Website', 'Support url', 'Support email', 'Metacritic url',
    'Score rank', 'Notes', 'Screenshots', 'Movies'
]

In [ ]:
dados_steam = dados_steam.drop(columns=colunas_para_remover)

In [ ]:
# Remove jogos com preço nulo ou gratuitos
coluna_preco = 'Price'
dados_steam = dados_steam.dropna(subset=[coluna_preco])
dados_steam = dados_steam[dados_steam[coluna_preco] > 0]

In [ ]:
print(f"Tamanho após a limpeza de dados da base: {len(dados_steam)} jogos.")

In [ ]:
tamanho_amostra = min(10000, len(dados_steam))
dados_steam = dados_steam.sample(n=tamanho_amostra, random_state=42)
dados_steam = dados_steam.reset_index(drop=True)

In [ ]:
print(f"Tamanho após limpeza e amostragem aleatória: {len(dados_steam)} jogos.")

# **Definição das Features**

In [ ]:
features_numericas = [
    'Peak CCU', 'Achievements', 'Positive', 'Negative',
    'Average playtime forever', 'Median playtime forever'
]

In [ ]:
features_categoricas = ['Genres', 'Windows', 'Mac', 'Linux']

In [ ]:
X = dados_steam[features_numericas + features_categoricas]
y = dados_steam[coluna_preco]

In [ ]:
print("Features (X) selecionadas prontas para o Pipeline!")
display(X.head())

In [ ]:
X_genres = X['Genres'].str.get_dummies(sep=',')

In [ ]:
X[['Windows', 'Mac', 'Linux']] = X[['Windows', 'Mac', 'Linux']].astype(int)

In [ ]:
X = pd.concat([X.drop(columns=['Genres']), X_genres], axis=1)

In [ ]:
print(f"Novo formato do X: {X.shape[0]} instâncias e {X.shape[1]} features numéricas prontas!")

# **Modelo 1: Regressão Linear**

In [ ]:
X_treino, X_teste, y_treino, y_teste = train_test_split(X, y, test_size=0.3, random_state=42)


In [ ]:
pipeline_lr = Pipeline([
    ('padronizacao', StandardScaler()),
    ('regressor', LinearRegression())
])

In [ ]:
#Treinamento da Regressão Linear
pipeline_lr.fit(X_treino, y_treino)
previsoes_lr = pipeline_lr.predict(X_teste)

In [ ]:
mse_lr = mean_squared_error(y_teste, previsoes_lr)
mae_lr = mean_absolute_error(y_teste, previsoes_lr)
r2_lr = r2_score(y_teste, previsoes_lr)

In [ ]:
print("\nRelatório de Regressão:")
print(f"R² (Coeficiente de Determinação): {r2_lr:.4f}")
print(f"MAE (Erro Absoluto Médio): {mae_lr:.2f}")
print(f"MSE (Erro Quadrático Médio): {mse_lr:.2f}")

# **Modelo 2: K-Nearest Neighbors (KNN)**

In [ ]:
pipeline_knn = Pipeline([
    ('padronizacao', StandardScaler()),
    ('regressor', KNeighborsRegressor())
])

In [ ]:
parametros_knn = {
    'regressor__n_neighbors': [3, 5, 7, 9],
    'regressor__weights': ['uniform', 'distance']
}

In [ ]:
#Treinamento do KNN
# Usei scoring='r2' para o GridSearch otimizar com base no Coeficiente de Determinação
grid_knn = GridSearchCV(pipeline_knn, param_grid=parametros_knn, cv=5, scoring='r2', n_jobs=-1)
grid_knn.fit(X_treino, y_treino)

In [ ]:
previsoes_knn = grid_knn.predict(X_teste)

In [ ]:
print(f"Melhores parâmetros KNN: {grid_knn.best_params_}")
print("\nRelatório de Regressão (KNN):")
print(f"R² (Coeficiente de Determinação): {r2_score(y_teste, previsoes_knn):.4f}")
print(f"MAE (Erro Absoluto Médio): {mean_absolute_error(y_teste, previsoes_knn):.2f}")
print(f"MSE (Erro Quadrático Médio): {mean_squared_error(y_teste, previsoes_knn):.2f}")

# **Modelo 3: Árvore de Decisão**

In [ ]:
pipeline_tree = Pipeline([
    ('padronizacao', StandardScaler()), # Árvores não exigem padronização, mas mantemos por consistência do Pipeline
    ('regressor', DecisionTreeRegressor(random_state=42))
])

In [ ]:
parametros_tree = {
    'regressor__max_depth': [None, 10, 20],
    'regressor__min_samples_split': [2, 5, 10]
}

In [ ]:
#Treinamento da Árvore de Decisão
grid_tree = GridSearchCV(pipeline_tree, param_grid=parametros_tree, cv=5, scoring='r2', n_jobs=-1)
grid_tree.fit(X_treino, y_treino)

In [ ]:
previsoes_tree = grid_tree.predict(X_teste)

In [ ]:
print(f"Melhores parâmetros Árvore: {grid_tree.best_params_}")
print("\nRelatório de Regressão (Árvore de Decisão):")
print(f"R² (Coeficiente de Determinação): {r2_score(y_teste, previsoes_tree):.4f}")
print(f"MAE (Erro Absoluto Médio): ${mean_absolute_error(y_teste, previsoes_tree):.2f}")
print(f"MSE (Erro Quadrático Médio): {mean_squared_error(y_teste, previsoes_tree):.2f}")

# **Modelo 4: Multilayer Perceptron (MLP)**

In [ ]:
pipeline_mlp = Pipeline([
    ('padronizacao', StandardScaler()),
    ('regressor', MLPRegressor(max_iter=500, random_state=42))
])

In [ ]:
parametros_mlp = {
    'regressor__hidden_layer_sizes': [(50,), (100,)],
    'regressor__activation': ['relu', 'tanh']
}

In [ ]:
#Treinamento do MLP
grid_mlp = GridSearchCV(pipeline_mlp, param_grid=parametros_mlp, cv=5, scoring='r2', n_jobs=-1)
grid_mlp.fit(X_treino, y_treino)

In [ ]:
previsoes_mlp = grid_mlp.predict(X_teste)

In [ ]:
print(f"Melhores parâmetros MLP: {grid_mlp.best_params_}")
print("\nRelatório de Regressão (MLP):")
print(f"R² (Coeficiente de Determinação): {r2_score(y_teste, previsoes_mlp):.4f}")
print(f"MAE (Erro Absoluto Médio): {mean_absolute_error(y_teste, previsoes_mlp):.2f}")
print(f"MSE (Erro Quadrático Médio): {mean_squared_error(y_teste, previsoes_mlp):.2f}")

# **Modelo 5: Support Vector Machine (SVM)**

In [ ]:
pipeline_svr = Pipeline([
    ('padronizacao', StandardScaler()),
    ('regressor', SVR())
])

In [ ]:
parametros_svr = {
    'regressor__kernel': ['linear', 'rbf'],
    'regressor__C': [0.1, 1.0]
}

In [ ]:
#Treinamento do SVM
grid_svr = GridSearchCV(pipeline_svr, param_grid=parametros_svr, cv=5, scoring='r2', n_jobs=-1)
grid_svr.fit(X_treino, y_treino)

In [ ]:
previsoes_svr = grid_svr.predict(X_teste)

In [ ]:
print(f"Melhores parâmetros SVR: {grid_svr.best_params_}")
print("\nRelatório de Regressão (SVR):")
print(f"R² (Coeficiente de Determinação): {r2_score(y_teste, previsoes_svr):.4f}")
print(f"MAE (Erro Absoluto Médio): {mean_absolute_error(y_teste, previsoes_svr):.2f}")
print(f"MSE (Erro Quadrático Médio): {mean_squared_error(y_teste, previsoes_svr):.2f}")

# **Comparação Entre os Modelos Treinados**

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
r2_scores = {
    'Regressão Linear': pipeline_lr.score(X_teste, y_teste),
    'KNN Regressor': grid_knn.score(X_teste, y_teste),
    'Árvore de Decisão': grid_tree.score(X_teste, y_teste),
    'MLP (Redes Neurais)': grid_mlp.score(X_teste, y_teste),
    'SVR': grid_svr.score(X_teste, y_teste)
}

In [ ]:
plt.figure(figsize=(12, 6))
grafico_r2 = sns.barplot(x=list(r2_scores.keys()), y=list(r2_scores.values()), hue=list(r2_scores.keys()), palette='viridis', legend=False)
plt.title('Comparação de R² (Coeficiente de Determinação) entre os Modelos', fontsize=14, pad=15)
plt.ylabel('R² no Conjunto de Teste', fontsize=12)
for i, v in enumerate(r2_scores.values()):
    # Se o valor for negativo, coloca o texto um pouco abaixo da barra
    offset = 0.02 if v >= 0 else -0.05
    grafico_r2.text(i, v + offset, f"{v:.4f}", ha='center', fontweight='bold', fontsize=11)

plt.tight_layout()
plt.show()

In [ ]:
modelos_regressao = {
    'Regressão Linear': pipeline_lr,
    'KNN Regressor': grid_knn,
    'Árvore de Decisão': grid_tree,
    'MLP (Redes Neurais)': grid_mlp,
    'SVR': grid_svr
}

In [ ]:
fig, axes = plt.subplots(nrows=2, ncols=3, figsize=(16, 10))
fig.suptitle('Desempenho dos Modelos: Valor Real vs. Valor Previsto ($)', fontsize=16, fontweight='bold', y=1.02)
axes = axes.flatten()
min_val = y_teste.min()
max_val = y_teste.max()
# Loop para plotar cada gráfico em um "quadradinho" da grade
for ax, (nome, modelo) in zip(axes, modelos_regressao.items()):
    # Faz a previsão com o modelo atual
    previsoes_atuais = modelo.predict(X_teste)

    # Plota a dispersão (pontos azuis)
    sns.scatterplot(x=y_teste, y=previsoes_atuais, alpha=0.5, ax=ax, color='#1f77b4')

    # Plota a linha de acerto perfeito (linha vermelha tracejada)
    ax.plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2)

    ax.set_title(nome, fontsize=12, fontweight='bold')
    ax.set_ylabel('Preço Previsto ($)')
    ax.set_xlabel('Preço Real ($)')

# O último "quadradinho" (índice 5) vai ficar vazio, então nós o escondemos
axes[5].axis('off')

plt.tight_layout()
plt.show()

# **Conclusão e Análise Crítica dos Resultados**

A estruturação rigorosa dos dados revelou a verdadeira natureza estatística do mercado: os modelos apresentaram $R^2$ próximo a zero ou negativo não por falha algorítmica, mas por ausência matemática de correlação linear entre as variáveis de engajamento e o preço.

*Inelasticidade da Variável Alvo:* O algoritmo provou que métricas pós-lançamento (tempo de jogo, conquistas, avaliações) não ditam o preço de prateleira. A precificação na Steam é uma decisão de negócio pré-lançamento, atrelada a variáveis ocultas neste dataset (orçamento de desenvolvimento, força da propriedade intelectual e idade do software).

*Convergência para a Média:* Diante da falta de padrões correlacionáveis, algoritmos como a Regressão Linear e o SVR otimizaram suas funções de perda prevendo simplesmente a média global de preços. Isso explica a linha horizontal perfeita nos gráficos de dispersão e o $R^2$ negativo (indicando que a tentativa de traçar uma reta de regressão foi matematicamente inferior a prever uma constante).